# PowerPoint 프레젠테이션 로더

Microsoft PowerPoint(`.pptx`, `.ppt`)는 프레젠테이션 제작에 가장 많이 사용되는 도구입니다.

이 노트북에서는 PowerPoint 파일에서 텍스트를 추출하여 LangChain Document로 변환하는 방법을 학습합니다.

## 주요 특징

- 슬라이드별 텍스트 추출
- 노트 영역 포함 가능
- Elements 모드로 요소별 분리

# 06. PowerPoint 프레젠테이션 로더

## 학습 목표
- `UnstructuredPowerPointLoader`로 `.pptx` 파일을 Document로 변환해요
- 기본 모드와 Elements 모드(요소별 분리)의 차이를 이해해요
- 슬라이드 텍스트를 RAG 시스템에서 활용하는 방법을 파악해요

## 사전 지식
- 00-Document-Loader 노트북 완료
- `pip install unstructured python-pptx` 설치

---

> **RAG 파이프라인 위치**: Load 단계 — 프레젠테이션 자료도 `Document`로 변환해서 검색 가능하게 만들어요.

## 환경 설정

In [3]:
from dotenv import load_dotenv


# 샘플 PowerPoint 파일 경로

# 아래에 코드를 작성하세요

load_dotenv()

FILE_PATH = "./data/sample.pptx"

## UnstructuredPowerPointLoader

`UnstructuredPowerPointLoader`는 PowerPoint 파일을 로드하는 표준 로더입니다.

### 특징

- `.pptx` 및 `.ppt` 파일 지원
- 슬라이드 텍스트 순서대로 추출
- 노트 영역 텍스트 포함 가능

### 필요한 패키지

```bash
pip install unstructured python-pptx
```

In [4]:
# ============================================================
# TODO: UnstructuredPowerPointLoader로 PowerPoint 파일을 로드해보세요
# 힌트: UnstructuredPowerPointLoader(FILE_PATH) → loader.load()
# 예상 결과: 로드된 문서 개수가 1개로 출력됩니다
# ============================================================

from langchain_community.document_loaders import UnstructuredPowerPointLoader

# 1단계: UnstructuredPowerPointLoader 생성
# - 기본 모드: 전체 슬라이드를 하나의 Document로 합쳐서 반환
loader = UnstructuredPowerPointLoader(FILE_PATH)

# 2단계: 문서 로드
docs = loader.load()


### 문서 내용 확인

In [7]:
# 첫 번째 문서의 내용 출력 (처음 300자)

# 아래에 코드를 작성하세요
docs[0].metadata
print(f' ==> [Line 4]: \033[38;2;163;73;142m[docs[0].metadata]\033[0m({type(docs[0].metadata).__name__}) = \033[38;2;95;43;161m{docs[0].metadata}\033[0m')

docs[0].page_content
print(f' ==> [Line 7]: \033[38;2;129;121;106m[docs[0].page_content]\033[0m({type(docs[0].page_content).__name__}) = \033[38;2;117;102;69m{docs[0].page_content}\033[0m')


 ==> [Line 4]: [docs[0].metadata](dict) = {'source': './data/sample.pptx'}
 ==> [Line 7]: [docs[0].page_content](str) = 청킹

LangChain 튜토리얼 - Chapter 7



01

CharacterTextSplitter

청킹 > CharacterTextSplitter



CharacterTextSplitter

6-2. Chunking — 가장 기본적인 텍스트 분할

3



학습 목표

청킹(Chunking)이 RAG에서 왜 필요한지 이해한다

CharacterTextSplitter의 단일 구분자 분할 방식을 익힌다

chunk_size, chunk_overlap, separator 파라미터의 동작 원리를 파악한다

4



RAG 파이프라인에서 청킹의 위치

6-1에서 로딩한 문서는 토큰 제한 때문에 그대로 LLM에 넣을 수 없다.

청킹은 문서를 LLM이 처리할 수 있는 크기로 나누는 작업이다.

5



CharacterTextSplitter란?

핵심 정의: 지정한 단일 구분자를 기준으로 텍스트를 나누는 가장 기본적인 분할 방식

단일 구분자 사용: 기본값 "\n\n" (이중 줄바꿈)으로 분할

문자 수 기반: len() 함수로 청크 크기를 측정

간단한 구조: 재귀 로직 없이 하나의 구분자만 사용

6



주요 파라미터

파라미터 기본값 설명 separator "\n\n" 텍스트를 분할할 구분자 chunk_size 4000 각 청크의 최대 문자 수 chunk_overlap 200 인접 청크 간 중복 문자 수 length_function len 텍스트 길이 계산 함수

7



코드 — 기본 사용법

python

from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(

    separator="\n\n",   # 이중 줄바

---

> **섹션 전환** — 기본 모드는 전체 슬라이드를 하나로 합쳐요. 슬라이드별로 분리하고 싶다면 아래의 `elements` 모드를 사용해 보세요.

> **실무 팁**: `metadata["page_number"]`와 `metadata["category"]`를 함께 활용하면 "몇 번째 슬라이드의 어떤 요소인지"를 정확히 추적할 수 있습니다. 이 정보를 LLM 응답에 포함시켜 출처를 명시할 수 있습니다.

## 핵심 정리 및 마무리

### 모드 선택 가이드

| 상황 | 추천 모드 |
|------|----------|
| 전체 내용을 한 번에 처리 | 기본 모드 |
| 슬라이드별 분석 필요 | `mode="elements"` |
| 제목/본문 구분이 필요 | `mode="elements"` |
| 간단한 텍스트 추출 | 기본 모드 |

### 주의 사항
> 이미지, 차트, 도형 안에 있는 텍스트는 추출되지 않아요. 시각 자료가 많은 프레젠테이션은 텍스트 추출 결과가 불완전할 수 있어요.

---

## 다음 예고

다음은 웹 페이지 콘텐츠를 불러오는 방법을 배워볼게요.

- **07-WebBase-Loader**: URL에서 직접 콘텐츠 로딩
- **08-TXT-Loader**: 텍스트 파일과 인코딩 처리

In [ ]:
# ============================================================
# TODO: mode="elements"로 PowerPoint를 요소별로 로드해보세요
# 힌트: UnstructuredPowerPointLoader(FILE_PATH, mode="elements")
# 예상 결과: 17개의 Document가 생성되고 각 요소의 category 메타데이터가 포함됩니다
# ============================================================

# 1단계: elements 모드로 로더 생성
# - mode="elements": 제목, 본문 등 각 요소를 별도 Document로 분리
# - 각 Document의 metadata에 category 정보 저장 (Title, NarrativeText 등)

# 2단계: 문서 로드
